# Quantitative chest computed tomography: regional differences in dual-energy-derived virtual vs. true non-contrast scans

Code accompanying the research article by:

Q.D. Strotzer, C. Schachner, L. Scheuermeyer, F. Raab, S. Meiler, M.V. Malfertheiner, C. Stroszczynski, O.W. Hamer

## Prerequisites

In [ ]:
import os
import SimpleITK as sitk
import pandas as pd
import numpy as np
from tqdm import tqdm
from lungmask import LMInferer
from nipype.interfaces.ants import Registration

path = "/Path/to/files/"
cases = ["List", "of", "case", "IDs"]

## Preprocessing

### Load, resample and convert to nifti

In [ ]:
reader = sitk.ImageSeriesReader()
reader.LoadPrivateTagsOn()

def resample(itk_image):
    original_spacing = itk_image.GetSpacing()
    original_size = itk_image.GetSize()

    out_size = [
        (original_size[0]),
        (original_size[1]),
        int(round(((original_size[2]) * (original_spacing[2] / 5)), 0)),
    ]

    out_spacing = [(original_spacing[0]), (original_spacing[1]), (5)]

    resample = sitk.ResampleImageFilter()
    resample.SetOutputSpacing(out_spacing)
    resample.SetSize(out_size)
    resample.SetOutputDirection(itk_image.GetDirection())
    resample.SetOutputOrigin(itk_image.GetOrigin())
    resample.SetTransform(sitk.Transform())
    resample.SetDefaultPixelValue(0)
    resample.SetInterpolator(
        sitk.sitkBSpline
    )

    return resample.Execute(itk_image)

def load(in_path, out_path, name):
    print(f"loading image: {in_path}/{name}")
    filenamesDICOM = reader.GetGDCMSeriesFileNames(in_path)
    reader.SetFileNames(filenamesDICOM)
    loadedImage = reader.Execute()
    loadedImage.SetOrigin((0, 0, 0))
    loadedImage_resampled = resample(loadedImage)
    sitk.WriteImage(loadedImage_resampled, os.path.join(out_path, name))

In [ ]:
failed = {}

for id in tqdm(cases):
    try:
        load(os.path.join(path, id, 'VNC'),
            os.path.join(path, id),
            f'{c}_5mm_VNC.nii.gz')

        load(os.path.join(path, id, 'TNC'),
            os.path.join(path, id),
            f'{c}_5mm_TNC.nii.gz')

    except Exception as e:
        failed[id] = e

for key, val in failed.items():
    print(key)

## Segment Lung
- using https://github.com/JoHof/lungmask

In [ ]:
def segment_lungs(img):
    inferer = LMInferer(modelname="R231")
    segmentation_array = inferer.apply(img)
    segmentation_image = sitk.GetImageFromArray(segmentation_array)
    segmentation_image.CopyInformation(img)

    return segmentation_image

def binarize(img):
    binary_filter = sitk.BinaryThresholdImageFilter()
    binary_filter.SetLowerThreshold(1)
    binary_filter.SetUpperThreshold(2)
    binary_filter.SetInsideValue(1)
    binary_filter.SetOutsideValue(0)

    return binary_filter.Execute(img)


def load_and_segment_lungs(path, id, img_name):
    image = sitk.ReadImage(
        os.path.join(path, id, f"{id}_{img_name}.nii.gz")
    )
    segmentation_image = segment_lungs(image)

    lsif = sitk.LabelShapeStatisticsImageFilter()
    lsif.Execute(binarize(segmentation_image))
    boundingBox = np.array(lsif.GetBoundingBox(1))

    # Crop to area of lung in z direction
    image_crop = image[
        :, :, boundingBox[2] : boundingBox[2] + boundingBox[5]
    ]
    segmentation_crop = segmentation_image[
        :, :, boundingBox[2] : boundingBox[2] + boundingBox[5]
    ]

    sitk.WriteImage(
        image_crop,
        os.path.join(path, id, f"{id}_{img_name}_cropped.nii.gz"),
    )
    sitk.WriteImage(
        segmentation_crop,
        os.path.join(
            path, id, f"{id}_{img_name}_lungsegmentation_cropped.nii.gz"
        ),
    )
    sitk.WriteImage(
        segmentation_image,
        os.path.join(
            path, id, f"{id}_{img_name}_lungsegmentation.nii.gz"
        ),
    )

## Co-Register VNC to TNC

In [ ]:
for id in tqdm(cases):
    reg = Registration()
    reg.inputs.fixed_image = os.path.join(path, id, f"{id}_5mm_TNC_cropped.nii.gz")
    reg.inputs.moving_image = os.path.join(
        path, id, f"{id}_5mm_VNC_cropped.nii.gz"
    )
    reg.inputs.output_transform_prefix = "transform"
    reg.inputs.transforms = ["Rigid", "Affine", "SyN"]
    reg.inputs.num_threads = 4
    reg.inputs.float = True
    reg.inputs.transform_parameters = [(2.0,), (2.0,), (0.25, 3.0, 0.0)]
    reg.inputs.number_of_iterations = [[1000, 200], [1000, 200], [100, 50, 30]]
    reg.inputs.dimension = 3
    reg.inputs.write_composite_transform = False
    reg.inputs.collapse_output_transforms = True
    reg.inputs.initialize_transforms_per_stage = False
    reg.inputs.output_inverse_warped_image = False
    reg.inputs.metric = ["Mattes"] * 3
    reg.inputs.metric_weight = [1] * 3
    reg.inputs.radius_or_number_of_bins = [32] * 3
    reg.inputs.sampling_strategy = ["Random", "Random", None]
    reg.inputs.sampling_percentage = [0.05, 0.05, None]
    reg.inputs.convergence_threshold = [1.0e-8, 1.0e-8, 1.0e-9]
    reg.inputs.convergence_window_size = [20] * 3
    reg.inputs.smoothing_sigmas = [[1, 0], [1, 0], [2, 1, 0]]
    reg.inputs.sigma_units = ["vox"] * 3
    reg.inputs.shrink_factors = [[2, 1], [2, 1], [3, 2, 1]]
    reg.inputs.use_estimate_learning_rate_once = [True] * 3
    reg.inputs.use_histogram_matching = [True] * 3
    reg.inputs.output_warped_image = os.path.join(
        path, id, f"{id}_5mm_VNC_cropped_warped.nii.gz"
    )

    reg.run()

### Change Pixel Type

In [ ]:
for id in tqdm(cases):

    img = sitk.ReadImage(
        os.path.join(path, id, f"{id}_5mm_VNC_cropped_warped.nii.gz"),
        sitk.sitkInt16,
    )

    sitk.WriteImage(
        img,
        os.path.join(path, id, f"{id}_5mm_VNC_cropped_warped_int.nii.gz"),
    )

## Manual Correction
- Segmentations and co-registrations need to be visually verified and corrected where needed!

# Keep largest Connected Component for Lobes

In [ ]:
def keep_largest_region_per_label(
    segmentation_image,
):
    output_image = sitk.Image(segmentation_image.GetSize(), segmentation_image.GetPixelID())
    output_image.CopyInformation(segmentation_image)

    labels = sitk.GetArrayViewFromImage(segmentation_image).flatten()
    unique_labels = set(labels) - {0}

    for label in unique_labels:

        binary_image = segmentation_image == label

        labeled_image = sitk.ConnectedComponent(binary_image)

        largest_component = sitk.LabelShapeStatisticsImageFilter()
        largest_component.Execute(labeled_image)
        largest_label = largest_component.GetNumberOfLabels()

        if largest_label > 0:
            largest_label_id = sorted([(largest_component.GetPhysicalSize(i), i) for i in range(1, largest_label + 1)], reverse=True)[0][1]
            largest_region_image = sitk.BinaryThreshold(labeled_image, largest_label_id, largest_label_id, int(label), 0)

            output_image = sitk.Add(output_image, largest_region_image)

    return output_image


for id in tqdm(cases):
    segmentation = sitk.ReadImage(
        os.path.join(
            path, id, f"{id}_5mm_..._lobesegmentation_cropped_corrected.nii.gz"
        )
    )

    seg_corrected = keep_largest_region_per_label(segmentation)

    sitk.WriteImage(
        seg_corrected,
        os.path.join(
            path,
            id,
            f"{id}_5mm_..._lobesegmentation_cropped_corrected_largestcomponents.nii.gz",
        ),
    )

## Create VOI Segmentations

In [ ]:
def set_mask_value_to_zero(image, value):
    array = sitk.GetArrayFromImage(image)
    array = np.where(array == value, 0, array)
    array = np.where(array > 0, 1, 0)
    imgneu = sitk.GetImageFromArray(array)
    imgneu.CopyInformation(image)

    return imgneu


def binary_segmentation(image, value):
    thres_filter = sitk.BinaryThresholdImageFilter()
    thres_filter.SetUpperThreshold(value)
    thres_filter.SetLowerThreshold(value)
    thres_filter.SetOutsideValue(0)
    thres_filter.SetInsideValue(1)

    return thres_filter.Execute(image)


def binarize_itkimage(
    img: sitk.Image,
    l_thresh: int,
    u_thresh: int,
):

    binary_filter = sitk.BinaryThresholdImageFilter()
    binary_filter.SetLowerThreshold(l_thresh)
    binary_filter.SetUpperThreshold(u_thresh)
    binary_filter.SetInsideValue(1)
    binary_filter.SetOutsideValue(0)

    return binary_filter.Execute(img)

### Return Segmentation of Center and Periphery

In [ ]:
def erode_image(img, erosion_mm):
    spacing = img.GetSpacing()
    num_pixels = [int(erosion_mm / spacing[dim]) for dim in range(img.GetDimension())]

    erodeFilter = sitk.BinaryErodeImageFilter()
    erodeFilter.SetKernelRadius(num_pixels)
    erodeFilter.SetForegroundValue(1)

    return erodeFilter.Execute(img)


def fill_between(side, eroded_img, full_img):
    seg1_np = sitk.GetArrayFromImage(eroded_img)
    seg2_np = sitk.GetArrayFromImage(full_img)

    output_np = np.zeros_like(seg1_np)
    dims = seg1_np.shape
    print(dims)

    for z in range(seg1_np.shape[0]):
        for y in range(seg1_np.shape[1]):
            # Identifying the boundaries in the X direction for both segmentations
            x_boundaries_seg1 = np.where(seg1_np[z, y, :] > 0)[0]
            x_boundaries_seg2 = np.where(seg2_np[z, y, :] > 0)[0]

            if (
                (side == "right")
                and (x_boundaries_seg1.size > 0)
                and (x_boundaries_seg2.size > 0)
            ):
                x_min = x_boundaries_seg1.min()
                x_max = x_boundaries_seg2.max()

                # Fill the space between the boundaries in the X direction
                output_np[z, y, x_min : x_max + 1] = 1

            if (
                (side == "left")
                and (x_boundaries_seg1.size > 0)
                and (x_boundaries_seg2.size > 0)
            ):
                x_min = x_boundaries_seg2.min()
                x_max = x_boundaries_seg1.max()

                # Fill the space between the boundaries in the X direction
                output_np[z, y, x_min : x_max + 1] = 1

    return output_np


def return_center_periphery_segmentation(img_r, img_l, erosion_mm=20.0):
    """
    Computes segmentation of lung center and periphery

    Args:
    - img_r: sitk.image that contains mask of right lung (label = 1)
    - img_l: sitk.image that contains mask of left lung (label = 1)

    Returns:
    - sitk.Image: Segmentation of lung center and periphery
        center right = 1
        center left = 2
        periphery right = 3
        periphery left = 4
    """

    eroded_img_r = erode_image(img_r, erosion_mm)
    eroded_img_l = erode_image(img_l, erosion_mm)

    filled_array_right = fill_between("right", eroded_img_r, img_r)
    filled_array_left = fill_between("left", eroded_img_l, img_l)

    filled_img_right = sitk.GetImageFromArray(filled_array_right)
    filled_img_right.CopyInformation(img_r)

    filled_img_left = sitk.GetImageFromArray(filled_array_left)
    filled_img_left.CopyInformation(img_l)

    result_image_re = 2 * img_r - filled_img_right
    result_image_re[result_image_re == 255] = 0
    result_image_re[result_image_re == 2] = 3

    result_image_li = 2 * img_l - filled_img_left
    result_image_li[result_image_li == 255] = 0
    result_image_li[result_image_li == 2] = 4
    result_image_li[result_image_li == 1] = 2

    return result_image_re + result_image_li

### Return Segmentation of Lower, Middle, and Upper Thirds by Volume

In [ ]:
def thirds_volume(image):
    array = sitk.GetArrayFromImage(image)
    binary = np.where(array > 0, 1, 0)

    total_volume = np.sum(binary)

    third_volume = total_volume // 3

    z_size = binary.shape[0]

    # start and end slice where actual volume starts
    for i in range(z_size):
        if np.any(binary[i]):
            start_index = i
            break

    for i in range(z_size - 1, -1, -1):
        if np.any(binary[i]):
            end_index = i
            break

    current_volume = 0
    split_indices = []

    for i in range(start_index, end_index + 1, 1):
        current_volume += np.sum(binary[i])

        if current_volume >= third_volume:
            split_indices.append(i)
            current_volume = 0

    thirds = np.copy(binary)

    thirds[: split_indices[0], :, :] = 1
    thirds[split_indices[0] : split_indices[1], :, :] = 2
    thirds[split_indices[1] :, :, :] = 3

    thirds = np.where(binary == 0, 0, thirds)

    thirds_image = sitk.GetImageFromArray(thirds.astype(np.uint8))
    thirds_image.CopyInformation(image)

    return thirds_image

### Return Segmentation of Ventral and Dorsal Half
- Labels: 1 = ventral; 2 = dorsal

In [ ]:
def ventral_dorsal_volume(image):
    array = sitk.GetArrayFromImage(image)
    binary = np.where(array > 0, 1, 0)

    total_volume = np.sum(binary)

    half_volume = total_volume // 2

    y_size = binary.shape[1]

    current_volume = 0
    split_index = None

    for i in range(y_size):
        current_volume += np.sum(binary[:, i, :])

        if current_volume >= half_volume:
            split_index = i
            break

    # assigning 1 to ventral half and 2 to dorsal half
    ventral_dorsal = np.copy(binary)
    ventral_dorsal[:, :split_index, :] = 1
    ventral_dorsal[:, split_index:, :] = 2

    # ensuring non-lung parts remain as 0
    ventral_dorsal = np.where(binary == 0, 0, ventral_dorsal)

    ventral_dorsal_image = sitk.GetImageFromArray(ventral_dorsal.astype(np.uint8))
    ventral_dorsal_image.CopyInformation(image)

    return ventral_dorsal_image

### Run VOI Segmentation 

In [ ]:
for id in tqdm(cases):
    os.makedirs(os.path.join(path, id, "segmentations"), exist_ok=True)

    segmentation = sitk.ReadImage(
        os.path.join(
            path,
            id,
            f"{id}_5mm_..._lobesegmentation_cropped_corrected_largestcomponents.nii.gz",
        )
    )
    seg_array = sitk.GetArrayFromImage(segmentation)
    segmentation_binary = binarize_itkimage(segmentation, l_thresh=1, u_thresh=2)
    sitk.WriteImage(
        segmentation_binary,
        os.path.join(path, id, "segmentations", f"{id}_binary.nii.gz"),
    )

    image_r = binarize_itkimage(segmentation, l_thresh=1, u_thresh=1)
    sitk.WriteImage(
        image_r, os.path.join(path, id, "segmentations", f"{id}_right.nii.gz")
    )

    image_l = binarize_itkimage(segmentation, l_thresh=2, u_thresh=2)
    sitk.WriteImage(
        image_l, os.path.join(path, id, "segmentations", f"{id}_left.nii.gz")
    )

    inner_outer_segmentation = return_center_periphery_segmentation(
        image_r, image_l
    )
    sitk.WriteImage(
        inner_outer_segmentation,
        os.path.join(path, id, "segmentations", f"{id}_inner_outer.nii.gz"),
    )

    thirds_segmentation = thirds_volume(segmentation_binary)
    sitk.WriteImage(
        thirds_segmentation,
        os.path.join(
            path, id, "segmentations", f"{id}_thirds.nii.gz"
        ),
    )

    ventral_dorsal_segmentation = ventral_dorsal_volume(segmentation_binary)
    sitk.WriteImage(
        ventral_dorsal_segmentation,
        os.path.join(
            path, id, "segmentations", f"{id}_ventral_dorsal.nii.gz"
        ),
    )

## Quantify

VOIs:   
- Whole Lung
- Thirds
- Center/Periphery
- Ventral/Dorsal

Parameter:    
- Vol [ml]
- Ref_Vol [%]
- MLD [HU] = Mean Lung Density
- LAV [%] = Volume% < -950 HU
- HAV [%] = Volume% -600 bis -250 HU
- 15th Percentile = HU Cutoff of the frequency distribution of HU numbers defining the lowest 15%

In [ ]:
def calc_perc15(array):
    hist, bins = np.histogram(array, bins=100)
    cdf = np.cumsum(hist)
    cdf = cdf / cdf[-1]  # normalize
    percentile = 15
    percentile_index = np.argmax(cdf >= percentile / 100)
    lowest_percentile = bins[percentile_index]
    
    return int(lowest_percentile)

def quantify(img, seg_array):
    img_array = sitk.GetArrayFromImage(img)

    space = img.GetSpacing()
    voxel = np.prod(space)

    vol = np.round(voxel * np.sum(seg_array[seg_array == 1]) / 1000, 2)

    mld = int(np.mean(img_array[seg_array == 1]))

    sd = int(np.std(img_array[seg_array == 1]))

    lav = np.round(
        voxel * np.sum(np.logical_and((seg_array == 1), (img_array < -950))) / 1000, 2
    )
    lav_perc = np.round(lav / vol * 100, 2)

    hav = np.round(
        voxel
        * np.sum(
            np.logical_and((seg_array == 1), (img_array >= -600), (img_array <= -250))
        )
        / 1000,
        2,
    )
    hav_perc = np.round(hav / vol * 100, 2)

    perc15 = calc_perc15(img_array[seg_array == 1])

    return {
        "Volume [ml]": vol,
        "MLD [HU]": mld,
        "SD [HU]": sd,
        "LAV < -950 HU [%]": lav_perc,
        "HAV > -600 to -250 HU [%]": hav_perc,
        "15th Percentile [HU]": perc15,
    }    

## Process Quantification

In [ ]:
results = []

thresh = ... #HU threshold for quantification

images = [("VNC...", []), ("TNC...", [])]

for id in tqdm(cases):
    segmentation = sitk.ReadImage(
        os.path.join(
            path,
            id,
            f"{id}_5mm_..._lobesegmentation_cropped_corrected_largestcomponents.nii.gz",
        )
    )
    seg_array = sitk.GetArrayFromImage(segmentation)
    segmentation_binary = binarize_itkimage(segmentation, l_thresh=1, u_thresh=5)

    image_r = sitk.ReadImage(
        os.path.join(path, id, "segmentations", f"{id}_right.nii.gz"),
    )
    image_r_array = sitk.GetArrayFromImage(image_r)

    image_l = sitk.ReadImage(
        os.path.join(path, id, "segmentations", f"{id}_left.nii.gz"),
    )
    image_l_array = sitk.GetArrayFromImage(image_l)

    inner_outer_segmentation = sitk.ReadImage(
        os.path.join(path, id, "segmentations", f"{id}_inner_outer.nii.gz"),
    )
    inner_outer_array = sitk.GetArrayFromImage(inner_outer_segmentation)

    thirds_segmentation = sitk.ReadImage(
        os.path.join(path, id, "segmentations", f"{id}_thirds.nii.gz"),
    )
    thirds_array = sitk.GetArrayFromImage(thirds_segmentation)

    ventral_dorsal_segmentation = sitk.ReadImage(
        os.path.join(path, id, "segmentations", f"{id}_ventral_dorsal.nii.gz"),
    )
    ventral_dorsal_array_gesamt = sitk.GetArrayFromImage(
        ventral_dorsal_segmentation
    )

    for img_name, res_list in images:

        result = {}

        image = sitk.ReadImage(os.path.join(path, id, f"{id}_{img}"))
        img_array = sitk.GetArrayFromImage(image)

        subsegmentations = [
            ("L", np.where((image_l_array == 1) & (img_array < thresh), 1, 0)),
            ("R", np.where((image_r_array == 1) & (img_array < thresh), 1, 0)),
            ("G", np.where((seg_array > 0) & (seg_array < 3) & (img_array < thresh), 1, 0)),
            ("GU", np.where((thirds_array == 3) & (img_array < thresh), 1, 0)),
            ("GM", np.where((thirds_array == 2) & (img_array < thresh), 1, 0)),
            ("GL", np.where((thirds_array == 1) & (img_array < thresh), 1, 0)),
            ("GC", np.where((inner_outer_array < 3) & (inner_outer_array > 0) & (img_array < thresh), 1, 0)),
            ("GP", np.where((inner_outer_array > 2) & (img_array < thresh), 1, 0)),
            ("GV", np.where((ventral_dorsal_array_gesamt == 1) & (img_array < thresh), 1, 0)),
            ("GD", np.where((ventral_dorsal_array_gesamt == 2) & (img_array < thresh), 1, 0)),
        ]

        for name, seg in subsegmentations:
            res = quantify(image, seg)
            res = {name + "_" + str(key): val for key, val in res.items()}
            result.update(res)


        result["ID"] = id

        res_list.append(result)

for img_name, res_list in images:
    df = pd.DataFrame.from_dict(res_list)
    col = df.pop("ID")
    df.insert(0, col.name, col)

    df.to_excel(os.path.join(path, f"Quantification_{img_name}.xlsx"))